In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.3 MLP feed-forward

After attention mixes information *across* tokens, the MLP processes each token
*independently*: expand to 4x width, a nonlinearity, project back. It is where
much of the model's capacity lives.

In [ ]:
from torch import nn

torch.manual_seed(0)
dim = 8
mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))
x = torch.randn(1, 4, dim)
out = mlp(x)
print("MLP preserves shape:", tuple(out.shape))

It is position-wise: the same vector gets the same output wherever it sits, so
the MLP never leaks information between positions, that is attention's job.

In [ ]:
same = torch.randn(dim)
x2 = torch.randn(1, 4, dim)
x2[0, 2] = same
assert tuple(out.shape) == (1, 4, dim)
assert torch.allclose(mlp(x2)[0, 2], mlp(same.view(1, 1, dim))[0, 0], atol=1e-6)